In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab; skipping Drive mount.")


In [ ]:
from pathlib import Path
import os


def resolve_data_path(filename: str) -> str:
    """
    Resolve data files in local Jupyter or Google Colab.
    In Colab, mount Google Drive first and place data under:
    /content/drive/MyDrive/feature-selection-experiments/data
    """
    candidates = []

    env_data_dir = os.environ.get("FEATURE_SELECTION_DATA_DIR")
    if env_data_dir:
        candidates.append(Path(env_data_dir) / filename)

    cwd = Path.cwd()
    candidates.extend([
        cwd / "data" / filename,
        cwd.parent / "data" / filename,
        Path("/content/data") / filename,
        Path("/content/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/data") / filename,
        Path(r"D:\feature-selection-experiments\data") / filename,
    ])

    for path in candidates:
        if path.exists():
            return str(path)

    search_roots = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.glob(f"**/{filename}"))
        for match in matches:
            if match.parent.name == "data":
                return str(match)
        if matches:
            return str(matches[0])

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n{checked}\n\n"
        "Upload the data folder to /content/data, or mount Google Drive and place it at "
        "/content/drive/MyDrive/feature-selection-experiments/data."
    )


DATA_DIR = Path(resolve_data_path("X_train.csv")).parent
if Path("/content/drive/MyDrive").exists():
    RESULT_DIR = Path("/content/drive/MyDrive/feature-selection-experiments/results/de-qlearning-spark")
else:
    RESULT_DIR = Path.cwd() / "results" / "de-qlearning-spark"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = RESULT_DIR


def output_path(filename: str) -> str:
    return str(RESULT_DIR / filename)


print(f"Using DATA_DIR: {DATA_DIR}")
print(f"Using RESULT_DIR: {RESULT_DIR}")


In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Tuple, List, Dict, Optional

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC
import time
from pyspark.sql import SparkSession

In [2]:
@dataclass
class QLDEConfig:
    population_size: int = 20
    max_generations: int = 30
    crossover_rate: float = 0.9
    init_F_min: float = 0.3
    init_F_max: float = 0.8
    q_learning_rate: float = 0.1   # alpha
    q_discount: float = 0.9        # gamma
    softmax_temperature: float = 1.0
    random_state: int = 42
    cv_splits: int = 5

    # Fitness = alpha_metric * macro_f1 - beta_size * feature_ratio
    alpha_metric: float = 0.95
    beta_size: float = 0.05

    # Các action điều chỉnh F
    actions: Tuple[float, ...] = (-0.05, 0.0, 0.05)

    # Ngưỡng biến continuous -> binary mask
    threshold: float = 0.5

    # Giới hạn F
    F_lower_bound: float = 0.1
    F_upper_bound: float = 1.0

    # Nếu cá thể chọn ít hơn min_features thì tự sửa
    min_features: int = 1

    # Dùng top ratio để cluster-guided mutation như paper
    top_ratio: float = 0.5
    val_size: float = 0.2

In [3]:
spark = (
    SparkSession.builder
    .appName("QLDE-Feature-Selection")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 15:17:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
def load_data(
    x_path: str,
    y_path: str
) -> Tuple[pd.DataFrame, np.ndarray]:
    X = pd.read_csv(x_path)
    y_df = pd.read_csv(y_path)

    if y_df.shape[1] != 1:
        raise ValueError("y_train.csv phải chỉ có đúng 1 cột nhãn.")

    y = y_df.iloc[:, 0].values

    # Encode nhãn nếu là chuỗi
    if y.dtype == object or str(y.dtype).startswith("string"):
        le = LabelEncoder()
        y = le.fit_transform(y)

    return X, y

In [5]:
def logistic_map_init(
    population_size: int,
    n_features: int,
    rng: np.random.Generator,
    mu: float = 3.99
) -> np.ndarray:
    """
    Khởi tạo quần thể bằng Logistic chaotic mapping.
    """
    pop = np.zeros((population_size, n_features), dtype=float)

    for i in range(population_size):
        phi = rng.uniform(0.001, 0.999)
        for j in range(n_features):
            phi = mu * phi * (1.0 - phi)
            pop[i, j] = phi

    return pop


def continuous_to_mask(
    vector: np.ndarray,
    threshold: float,
    min_features: int,
    rng: np.random.Generator
) -> np.ndarray:
    """
    Convert continuous vector [0,1] -> binary mask.
    """
    mask = (vector > threshold).astype(int)

    if mask.sum() < min_features:
        # ép chọn ngẫu nhiên min_features cột
        idx = rng.choice(len(mask), size=min_features, replace=False)
        mask[idx] = 1

    return mask



def hamming_distance_ratio(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Return the normalized Hamming distance between two binary masks."""
    mask_a = np.asarray(mask_a).astype(int)
    mask_b = np.asarray(mask_b).astype(int)
    if mask_a.shape != mask_b.shape:
        raise ValueError("Two masks must have the same shape.")
    return float(np.mean(mask_a != mask_b))


def continuous_population_to_masks(
    population: np.ndarray,
    threshold: float = 0.5,
    min_features: int = 1,
    use_sigmoid: bool = False
) -> np.ndarray:
    """Convert a continuous population to binary feature masks for metric tracking."""
    values = 1.0 / (1.0 + np.exp(-population)) if use_sigmoid else np.asarray(population)
    masks = (values > threshold).astype(int)

    if min_features > 0:
        for i in range(masks.shape[0]):
            if masks[i].sum() < min_features:
                top_idx = np.argsort(values[i])[-min_features:]
                masks[i, top_idx] = 1

    return masks


def compute_population_diversity(population_masks: np.ndarray) -> float:
    """Average pairwise Hamming distance among all masks in the current population."""
    population_masks = np.asarray(population_masks).astype(int)
    n_individuals = population_masks.shape[0]
    if n_individuals < 2:
        return 0.0

    total_distance = 0.0
    pair_count = 0
    for i in range(n_individuals):
        for j in range(i + 1, n_individuals):
            total_distance += hamming_distance_ratio(population_masks[i], population_masks[j])
            pair_count += 1

    return float(total_distance / pair_count)


def compute_exploration_exploitation(
    new_masks: np.ndarray,
    previous_masks: np.ndarray,
    similarity_threshold: float = 0.1
) -> Tuple[float, float]:
    """
    Exploration is the fraction of new masks whose nearest previous neighbor
    is farther than the threshold. Exploitation is the complementary fraction.
    """
    new_masks = np.asarray(new_masks).astype(int)
    previous_masks = np.asarray(previous_masks).astype(int)
    if len(new_masks) == 0 or len(previous_masks) == 0:
        return 0.0, 0.0

    exploration_count = 0
    for new_mask in new_masks:
        nearest_distance = min(
            hamming_distance_ratio(new_mask, previous_mask)
            for previous_mask in previous_masks
        )
        if nearest_distance > similarity_threshold:
            exploration_count += 1

    exploration = exploration_count / len(new_masks)
    exploitation = 1.0 - exploration
    return float(exploration), float(exploitation)

def select_action_softmax(
    q_table: np.ndarray,
    state: int,
    temperature: float,
    rng: np.random.Generator
) -> int:
    """
    Chọn action theo softmax trên hàng Q[state].
    state: 0 hoặc 1
    """
    q_values = q_table[state].copy()

    if temperature <= 0:
        return int(np.argmax(q_values))

    q_scaled = q_values / temperature
    q_scaled = q_scaled - np.max(q_scaled)
    probs = np.exp(q_scaled)
    probs = probs / probs.sum()

    return int(rng.choice(len(q_values), p=probs))


def evaluate_feature_subset(
    X: pd.DataFrame,
    y: np.ndarray,
    mask: np.ndarray,
    config: QLDEConfig
) -> Tuple[float, float, int]:
    """
    Evaluate một tập feature bằng 1 lần chia train/validation.
    Trả về:
    - fitness
    - macro_f1 trên validation
    - số lượng feature được chọn
    """
    selected_cols = X.columns[mask == 1]
    X_sel = X[selected_cols]

    if X_sel.shape[1] == 0:
        return -1e9, 0.0, 0

    X_train_fs, X_val_fs, y_train_fs, y_val_fs = train_test_split(
        X_sel,
        y,
        test_size=config.val_size,
        random_state=config.random_state,
        stratify=y
    )

    clf = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=config.random_state)

    clf.fit(X_train_fs, y_train_fs)
    y_pred = clf.predict(X_val_fs)

    macro_f1 = f1_score(y_val_fs, y_pred, average="macro")
    feature_ratio = X_sel.shape[1] / X.shape[1]

    fitness = (
        config.alpha_metric * macro_f1
        - config.beta_size * feature_ratio
    )

    return fitness, macro_f1, X_sel.shape[1]

class DummyConfig:
    def __init__(self, d):
        self.__dict__ = d
        
def spark_eval_initial(task, X_bc, cols_bc, y_bc, config_bc):
    i, individual, seed = task

    rng = np.random.default_rng(seed)
    X_df = pd.DataFrame(X_bc.value, columns=cols_bc.value)
    y = y_bc.value
    config = DummyConfig(config_bc.value)

    mask = continuous_to_mask(
        individual,
        config.threshold,
        config.min_features,
        rng
    )
    fit_i, f1_i, nf_i = evaluate_feature_subset(X_df, y, mask, config)

    return i, mask, fit_i, f1_i, nf_i


def spark_build_and_eval_trial(
    task,
    X_bc, cols_bc, y_bc, config_bc,
    pop_bc, fit_bc, F_bc, elite_bc, q_bc
):
    i, seed = task

    rng = np.random.default_rng(seed)

    X_df = pd.DataFrame(X_bc.value, columns=cols_bc.value)
    y = y_bc.value
    config = DummyConfig(config_bc.value)

    population = pop_bc.value
    fitnesses = fit_bc.value
    F_values = F_bc.value
    elite_indices = elite_bc.value
    q_table = q_bc.value

    n_features = population.shape[1]

    parent = population[i].copy()
    parent_fitness = fitnesses[i]

    current_state = 1

    action_idx = select_action_softmax(
        q_table=q_table,
        state=current_state,
        temperature=config.softmax_temperature,
        rng=rng
    )
    delta_F = config.actions[action_idx]

    F_i = F_values[i] + delta_F
    F_i = float(np.clip(F_i, config.F_lower_bound, config.F_upper_bound))

    Lk = int(rng.choice(elite_indices))

    candidates = [idx for idx in range(len(population)) if idx != i]
    r1, r2 = rng.choice(candidates, size=2, replace=False)

    mutant = population[Lk] + F_i * (population[r1] - population[r2])
    mutant = np.clip(mutant, 0.0, 1.0)

    trial = parent.copy()
    j_rand = rng.integers(0, n_features)

    for j in range(n_features):
        if rng.random() <= config.crossover_rate or j == j_rand:
            trial[j] = mutant[j]

    trial = np.clip(trial, 0.0, 1.0)

    trial_mask = continuous_to_mask(
        trial,
        config.threshold,
        config.min_features,
        rng
    )

    trial_fitness, trial_f1, trial_nf = evaluate_feature_subset(
        X_df, y, trial_mask, config
    )

    return {
        "i": i,
        "trial": trial,
        "trial_mask": trial_mask,
        "trial_fitness": trial_fitness,
        "trial_f1": trial_f1,
        "trial_nf": trial_nf,
        "parent_fitness": parent_fitness,
        "action_idx": action_idx,
        "current_state": current_state,
        "F_i": F_i,
    }
    
class QLDEFeatureSelector:
    def __init__(self, config: Optional[QLDEConfig] = None, spark_context=None):
        self.config = config if config is not None else QLDEConfig()
        self.rng = np.random.default_rng(self.config.random_state)
        self.sc = spark_context

        self.best_mask_: Optional[np.ndarray] = None
        self.best_fitness_: float = -np.inf
        self.best_macro_f1_: float = 0.0
        self.best_num_features_: int = 0
        self.history_: List[Dict] = []
        self.optimization_metrics_: Optional[pd.DataFrame] = None
        self.q_table_: Optional[np.ndarray] = None
        self.feature_selection_time_: float = 0.0

    def _initialize_population(self, n_features: int) -> Tuple[np.ndarray, np.ndarray]:
        population = logistic_map_init(
            population_size=self.config.population_size,
            n_features=n_features,
            rng=self.rng
        )

        # F riêng cho từng cá thể như paper
        F_values = self.rng.uniform(
            self.config.init_F_min,
            self.config.init_F_max,
            size=self.config.population_size
        )

        return population, F_values

    def fit(self, X_bc, y_bc, cols_bc):
        import time
    
        fs_start = time.perf_counter()
    
        # Lấy dữ liệu từ broadcast
        X_values = X_bc.value
        y = y_bc.value
        columns = list(cols_bc.value)
    
        # DataFrame đầy đủ để dùng cho branch không Spark
        X_df_full = pd.DataFrame(X_values, columns=columns)
    
        n_features = X_values.shape[1]
    
        # Q-table: 2 states x num_actions
        q_table = np.zeros((2, len(self.config.actions)), dtype=float)
    
        population, F_values = self._initialize_population(n_features)

        fitnesses = np.zeros(self.config.population_size)
        macro_f1s = np.zeros(self.config.population_size)
        num_features = np.zeros(self.config.population_size, dtype=int)
    
        # Broadcast config dùng xuyên suốt
        config_bc = self.sc.broadcast(self.config.__dict__) if self.sc is not None else None    

        # =========================
        # 1) ĐÁNH GIÁ QUẦN THỂ BAN ĐẦU
        # =========================
        init_eval_start = time.perf_counter()
    
        if self.sc is not None:
            init_tasks = [
                (i, population[i].copy(), self.config.random_state + 1000 + i)
                for i in range(self.config.population_size)
            ]
    
            init_results = (
                self.sc.parallelize(init_tasks, numSlices=min(self.config.population_size, 8))
                .map(lambda task: spark_eval_initial(task, X_bc, cols_bc, y_bc, config_bc))
                .collect()
            )
    
            for i, mask, fit_i, f1_i, nf_i in sorted(init_results, key=lambda x: x[0]):
                fitnesses[i] = fit_i
                macro_f1s[i] = f1_i
                num_features[i] = nf_i
    
                if fit_i > self.best_fitness_:
                    self.best_fitness_ = fit_i
                    self.best_mask_ = mask.copy()
                    self.best_macro_f1_ = f1_i
                    self.best_num_features_ = nf_i
        else:
            for i in range(self.config.population_size):
                mask = continuous_to_mask(
                    population[i],
                    self.config.threshold,
                    self.config.min_features,
                    self.rng
                )
                fit_i, f1_i, nf_i = evaluate_feature_subset(
                    X_df_full, y, mask, self.config
                )
                fitnesses[i] = fit_i
                macro_f1s[i] = f1_i
                num_features[i] = nf_i
    
                if fit_i > self.best_fitness_:
                    self.best_fitness_ = fit_i
                    self.best_mask_ = mask.copy()
                    self.best_macro_f1_ = f1_i
                    self.best_num_features_ = nf_i
    
        init_eval_time = time.perf_counter() - init_eval_start
        print(f"[Timing] Initial population evaluation: {init_eval_time:.4f} seconds")

        # =========================
        # 2) VÒNG LẶP CHÍNH
        # =========================
        for gen in range(self.config.max_generations):
            gen_start = time.perf_counter()
    
            previous_masks_for_metrics = continuous_population_to_masks(
                population,
                threshold=self.config.threshold,
                min_features=self.config.min_features
            )
            trial_masks_for_metrics = np.zeros_like(previous_masks_for_metrics)

            new_population = population.copy()
            new_fitnesses = fitnesses.copy()
            new_macro_f1s = macro_f1s.copy()
            new_num_features = num_features.copy()
    
            sorted_idx = np.argsort(-fitnesses)
            top_k = max(1, int(self.config.top_ratio * self.config.population_size))
            elite_indices = sorted_idx[:top_k]
    
            gen_improvements = 0
            eval_start = time.perf_counter()
    
            if self.sc is not None:
                # Broadcast các dữ liệu lớn theo từng generation
                pop_bc = self.sc.broadcast(population)
                fit_bc = self.sc.broadcast(fitnesses)
                F_bc = self.sc.broadcast(F_values)
                elite_bc = self.sc.broadcast(elite_indices)
                q_bc = self.sc.broadcast(q_table)
    
                tasks = [
                    (i, self.config.random_state + (gen + 1) * 100000 + i)
                    for i in range(self.config.population_size)
                ]

                results = (
                    self.sc.parallelize(tasks, numSlices=min(self.config.population_size, 8))
                    .map(lambda task: spark_build_and_eval_trial(
                        task,
                        X_bc, cols_bc, y_bc, config_bc,
                        pop_bc, fit_bc, F_bc, elite_bc, q_bc
                    ))
                    .collect()
                )
    
                # Giải phóng broadcast từng generation
                pop_bc.unpersist()
                fit_bc.unpersist()
                F_bc.unpersist()
                elite_bc.unpersist()
                q_bc.unpersist()
    
                results = sorted(results, key=lambda x: x["i"])
    
                # Update tuần tự ở driver
                for res in results:
                    i = res["i"]
                    trial = res["trial"]
                    trial_mask = res["trial_mask"]
                    trial_fitness = res["trial_fitness"]
                    trial_masks_for_metrics[i] = trial_mask
                    trial_f1 = res["trial_f1"]
                    trial_nf = res["trial_nf"]
                    parent_fitness = res["parent_fitness"]
                    action_idx = res["action_idx"]
                    current_state = res["current_state"]
                    F_i = res["F_i"]
    
                    if trial_fitness > parent_fitness:
                        reward = 1.0
                        next_state = 0
                    else:
                        reward = 0.0
                        next_state = 1
    
                    best_next_q = np.max(q_table[next_state])
                    old_q = q_table[current_state, action_idx]
                    q_table[current_state, action_idx] = old_q + self.config.q_learning_rate * (
                        reward + self.config.q_discount * best_next_q - old_q
                    )
    
                    if trial_fitness > parent_fitness:
                        new_population[i] = trial
                        new_fitnesses[i] = trial_fitness
                        new_macro_f1s[i] = trial_f1
                        new_num_features[i] = trial_nf
                        F_values[i] = F_i
                        gen_improvements += 1

                        if trial_fitness > self.best_fitness_:
                            self.best_fitness_ = trial_fitness
                            self.best_mask_ = trial_mask.copy()
                            self.best_macro_f1_ = trial_f1
                            self.best_num_features_ = trial_nf
    
            else:
                for i in range(self.config.population_size):
                    parent = population[i].copy()
                    parent_fitness = fitnesses[i]
    
                    current_state = 1
    
                    action_idx = select_action_softmax(
                        q_table=q_table,
                        state=current_state,
                        temperature=self.config.softmax_temperature,
                        rng=self.rng
                    )
                    delta_F = self.config.actions[action_idx]
    
                    F_i = F_values[i] + delta_F
                    F_i = float(np.clip(
                        F_i,
                        self.config.F_lower_bound,
                        self.config.F_upper_bound
                    ))

                    Lk = int(self.rng.choice(elite_indices))
                    candidates = [idx for idx in range(self.config.population_size) if idx != i]
                    r1, r2 = self.rng.choice(candidates, size=2, replace=False)
    
                    mutant = population[Lk] + F_i * (population[r1] - population[r2])
                    mutant = np.clip(mutant, 0.0, 1.0)
    
                    trial = parent.copy()
                    j_rand = self.rng.integers(0, n_features)
    
                    for j in range(n_features):
                        if self.rng.random() <= self.config.crossover_rate or j == j_rand:
                            trial[j] = mutant[j]
    
                    trial = np.clip(trial, 0.0, 1.0)
    
                    trial_mask = continuous_to_mask(
                        trial,
                        self.config.threshold,
                        self.config.min_features,
                        self.rng
                    )
                    trial_fitness, trial_f1, trial_nf = evaluate_feature_subset(
                        X_df_full, y, trial_mask, self.config
                    )
                    trial_masks_for_metrics[i] = trial_mask
    
                    if trial_fitness > parent_fitness:
                        reward = 1.0
                        next_state = 0
                    else:
                        reward = 0.0
                        next_state = 1

                    best_next_q = np.max(q_table[next_state])
                    old_q = q_table[current_state, action_idx]
                    q_table[current_state, action_idx] = old_q + self.config.q_learning_rate * (
                        reward + self.config.q_discount * best_next_q - old_q
                    )
    
                    if trial_fitness > parent_fitness:
                        new_population[i] = trial
                        new_fitnesses[i] = trial_fitness
                        new_macro_f1s[i] = trial_f1
                        new_num_features[i] = trial_nf
                        F_values[i] = F_i
                        gen_improvements += 1
    
                        if trial_fitness > self.best_fitness_:
                            self.best_fitness_ = trial_fitness
                            self.best_mask_ = trial_mask.copy()
                            self.best_macro_f1_ = trial_f1
                            self.best_num_features_ = trial_nf
    
            eval_time = time.perf_counter() - eval_start

            population = new_population
            fitnesses = new_fitnesses
            macro_f1s = new_macro_f1s
            num_features = new_num_features

            current_masks_for_metrics = continuous_population_to_masks(
                population,
                threshold=self.config.threshold,
                min_features=self.config.min_features
            )
            diversity = compute_population_diversity(current_masks_for_metrics)
            exploration_threshold = getattr(self.config, "exploration_threshold", 0.1)
            exploration, exploitation = compute_exploration_exploitation(
                new_masks=trial_masks_for_metrics,
                previous_masks=previous_masks_for_metrics,
                similarity_threshold=exploration_threshold
            )
    
            self.history_.append({
                "generation": gen + 1,
                "best_fitness": float(np.max(fitnesses)),
                "mean_fitness": float(np.mean(fitnesses)),
                "best_macro_f1": float(np.max(macro_f1s)),
                "mean_macro_f1": float(np.mean(macro_f1s)),
                "best_num_features": int(num_features[np.argmax(fitnesses)]),
                "improvements": gen_improvements,
                "diversity": diversity,
                "exploration": exploration,
                "exploitation": exploitation,
                "exploration_threshold": exploration_threshold,
                "eval_time_sec": eval_time
            })
    
            gen_time = time.perf_counter() - gen_start
    
            print(
                f"Generation {gen+1:02d} | "
                f"Best fitness = {np.max(fitnesses):.5f} | "
                f"Best macro-F1 = {np.max(macro_f1s):.5f} | "
                f"Selected features = {num_features[np.argmax(fitnesses)]} | "
                f"Improvements = {gen_improvements} | "
                f"Diversity = {diversity:.4f} | "
                f"Exploration = {exploration:.4f} | "
                f"Exploitation = {exploitation:.4f} | "
                f"Eval time = {eval_time:.4f}s | "
                f"Total gen time = {gen_time:.4f}s"
            )
    
        self.optimization_metrics_ = pd.DataFrame(self.history_)
        self.optimization_metrics_.to_csv(output_path("optimization_metrics_de_qlearning_spark_with_formulas.csv"), index=False)
        print("[Metrics] Saved optimization_metrics_de_qlearning_spark_with_formulas.csv")
        self.q_table_ = q_table
        self.feature_selection_time_ = time.perf_counter() - fs_start
        print(f"[Timing] Total feature selection time: {self.feature_selection_time_:.4f} seconds")
    
        if config_bc is not None:
            config_bc.unpersist()
    
        return self

    def get_selected_features(self, feature_names: List[str]) -> List[str]:
        if self.best_mask_ is None:
            raise ValueError("Model chưa fit.")
        return [f for f, m in zip(feature_names, self.best_mask_) if m == 1]

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.best_mask_ is None:
            raise ValueError("Model chưa fit.")
        selected_cols = X.columns[self.best_mask_ == 1]
        return X[selected_cols]

    def fit_transform(self, X: pd.DataFrame, y: np.ndarray) -> pd.DataFrame:
        self.fit(X, y)
        return self.transform(X)

In [6]:
if __name__ == "__main__":
    # Đổi lại đúng tên file của bạn
    X_path = resolve_data_path("X_train.csv")
    y_path = resolve_data_path("y_train.csv")

    X, y = load_data(X_path, y_path)
    X_bc = spark.sparkContext.broadcast(X.values)
    y_bc = spark.sparkContext.broadcast(y)
    cols_bc = spark.sparkContext.broadcast(list(X.columns))


    config = QLDEConfig(
        population_size=20,
        max_generations=100,
        crossover_rate=0.9,
        init_F_min=0.3,
        init_F_max=0.8,
        q_learning_rate=0.1,
        q_discount=0.9,
        softmax_temperature=1.0,
        cv_splits=5,
        alpha_metric=0.95,
        beta_size=0.05,
        actions=(-0.05, 0.0, 0.05),
        threshold=0.5,
        min_features=1,
        random_state=42
    )

    spark = (
        SparkSession.builder
        .appName("QLDE-Feature-Selection")
        .master("local[*]")
        .getOrCreate()
    )
    
    selector = QLDEFeatureSelector(config=config, spark_context=spark.sparkContext)
    selector.fit(X_bc, y_bc, cols_bc)
    
    print("Total FS time:", selector.feature_selection_time_)

    selected_features = selector.get_selected_features(list(X.columns))
    X_selected = selector.transform(X)

    print("\n========== KẾT QUẢ CUỐI ==========")
    print(f"Best fitness       : {selector.best_fitness_:.6f}")
    print(f"Best macro-F1      : {selector.best_macro_f1_:.6f}")
    print(f"Số feature đã chọn : {selector.best_num_features_}")
    print(f"Tên feature chọn   : {selected_features}")

    print("\nQ-table cuối:")
    print(selector.q_table_)

    # Lưu file kết quả
    X_selected.to_csv(output_path("X_train_selected.csv"), index=False)
    pd.DataFrame({"selected_feature": selected_features}).to_csv(
        output_path("selected_features_list.csv"), index=False
    )
    pd.DataFrame(selector.history_).to_csv(output_path("qlde_history.csv"), index=False)

    print("\nĐã lưu:")
    print("- X_train_selected.csv")
    print("- selected_features_list.csv")
    print("- qlde_history.csv")

[Timing] Initial population evaluation: 25.7018 seconds


Generation 01 | Best fitness = 0.68839 | Best macro-F1 = 0.75256 | Selected features = 531 | Improvements = 13 | Eval time = 24.3036s | Total gen time = 24.3068s


Generation 02 | Best fitness = 0.68839 | Best macro-F1 = 0.75256 | Selected features = 531 | Improvements = 12 | Eval time = 23.3160s | Total gen time = 23.3161s


Generation 03 | Best fitness = 0.68839 | Best macro-F1 = 0.75256 | Selected features = 531 | Improvements = 5 | Eval time = 23.3348s | Total gen time = 23.3349s


Generation 04 | Best fitness = 0.68839 | Best macro-F1 = 0.75256 | Selected features = 531 | Improvements = 8 | Eval time = 23.6761s | Total gen time = 23.6762s


Generation 05 | Best fitness = 0.68839 | Best macro-F1 = 0.75256 | Selected features = 531 | Improvements = 8 | Eval time = 22.7319s | Total gen time = 22.7321s


... (91 outputs left out)

Generation 97 | Best fitness = 0.73071 | Best macro-F1 = 0.79685 | Selected features = 526 | Improvements = 3 | Eval time = 20.6888s | Total gen time = 20.6891s


Generation 98 | Best fitness = 0.73071 | Best macro-F1 = 0.79685 | Selected features = 526 | Improvements = 2 | Eval time = 20.7263s | Total gen time = 20.7264s


Generation 99 | Best fitness = 0.73071 | Best macro-F1 = 0.79685 | Selected features = 526 | Improvements = 3 | Eval time = 21.4821s | Total gen time = 21.4823s


Generation 100 | Best fitness = 0.73071 | Best macro-F1 = 0.79685 | Selected features = 526 | Improvements = 2 | Eval time = 21.3003s | Total gen time = 21.3006s
[Timing] Total feature selection time: 2169.1350 seconds
Total FS time: 2169.134961902

========== KẾT QUẢ CUỐI ==========
Best fitness       : 0.730708
Best macro-F1      : 0.796850
Số feature đã chọn : 526
Tên feature chọn   : ['AAK1', 'ABCA1', 'AC090498.1', 'ACAA1', 'ACAP1', 'ACP5', 'ACSL4', 'ADAM15', 'AGPAT2', 'AGR2', 'AGTRAP', 'ALCAM', 'ALDOA', 'ANAPC11', 'ANXA2', 'ANXA4', 'AP1S1', 'AP2S1', 'AP3D1', 'AP3S1', 'APOE', 'APP', 'ARF4', 'ARF6', 'ARL2', 'ARL4A', 'ARL8B', 'ARPC1A', 'ARPC2', 'ATP11A', 'ATP1B3', 'ATP5H', 'ATP6V0D1', 'ATP6V1A', 'AUP1', 'AURKAIP1', 'AVPI1', 'B2M', 'B4GALT1', 'BAG1', 'BCAS2', 'BID', 'BIN1', 'BLVRA', 'BLVRB', 'BNIP3', 'BNIP3L', 'BSG', 'BST2', 'BTG1', 'BTG2', 'BUD31', 'C11orf31', 'C15orf48', 'C17orf89', 'C19orf33', 'C19orf60', 'C1QA', 'C5AR1', 'C6orf48', 'C8orf4', 'CALM2', 'CALR', 'CANX', 'CAPG', 'CBR1'

In [7]:
import pandas as pd

# Đọc dữ liệu
X_train = pd.read_csv(resolve_data_path("X_train.csv"))
X_val = pd.read_csv(resolve_data_path("X_val.csv"))
X_test = pd.read_csv(resolve_data_path("X_test.csv"))

y_train = pd.read_csv(resolve_data_path("y_train.csv")).iloc[:, 0].values
y_val = pd.read_csv(resolve_data_path("y_val.csv")).iloc[:, 0].values
y_test = pd.read_csv(resolve_data_path("y_test.csv")).iloc[:, 0].values

# Áp feature selected lên cả 3 tập
X_train_selected = X_train[selected_features].copy()
X_val_selected = X_val[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

print("Train selected shape:", X_train_selected.shape)
print("Val selected shape:", X_val_selected.shape)
print("Test selected shape:", X_test_selected.shape)

Train selected shape: (16000, 526)
Val selected shape: (2000, 526)
Test selected shape: (2000, 526)


In [8]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    row = {"model": model_name}
    for cls in CLASS_NAMES:
        for m in ["precision","recall","f1-score"]:
            col = f"{cls}_{m.replace('-score','')}"
            row[col] = round(report[cls][m], 4)
    for avg in ["macro avg","weighted avg"]:
        prefix = avg.replace(" avg","")
        for m in ["precision","recall","f1-score"]:
            col = f"{prefix}_{m.replace('-score','')}"
            row[col] = round(report[avg][m], 4)
    row["accuracy"] = round(report["accuracy"], 4)
    return row

In [9]:
all_results = [] 

from sklearn.neighbors         import KNeighborsClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.naive_bayes       import GaussianNB
from sklearn.linear_model      import LogisticRegression
from sklearn.svm               import LinearSVC, SVC
from sklearn.model_selection   import GridSearchCV
from sklearn.pipeline          import Pipeline
import numpy as np

unique_classes = np.unique(y_train)
N_CLASSES = len(unique_classes)
CLASS_NAMES = [str(c) for c in unique_classes]

print("N_CLASSES :", N_CLASSES)
print("CLASS_NAMES:", CLASS_NAMES)

ML_CONFIGS = [
    (
        "KNN",
        KNeighborsClassifier(),
        {"n_neighbors":[3,5,7,11], "weights":["uniform","distance"],
         "metric":["euclidean","manhattan"]}
    ),
    (
        "DecisionTree",
        DecisionTreeClassifier(random_state=42),
        {"max_depth":[None,10,20,30], "min_samples_split":[2,5,10],
         "criterion":["gini","entropy"]}
    ),
    (
        "RandomForest",
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators":[100,200], "max_depth":[None,10,20],
         "min_samples_split":[2,5]}
    ),
    (
        "GaussianNB",
        GaussianNB(),
        {"var_smoothing":[1e-9,1e-8,1e-7,1e-6]}
    ),
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
        {"C":[0.01,0.1,1,10], "solver":["lbfgs","saga"],
         "multi_class":["multinomial"]}
    ),
    (
        "LinearSVM",
        LinearSVC(max_iter=2000, random_state=42),
        {"C":[0.01,0.1,1,10]}
    ),
    (
        "RBF_SVM",
        SVC(kernel="rbf", random_state=42),
        {"C":[0.1,1,10], "gamma":["scale","auto"]}
    ),
]

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

all_results = []
report_rows = []

for name, estimator, param_grid in ML_CONFIGS:
    print(f"\n[{name}] GridSearch đang chạy...")

    gs = GridSearchCV(
        estimator,
        param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0
    )

    gs.fit(X_train_selected, y_train)

    best = gs.best_estimator_
    y_pred = best.predict(X_test_selected)

    print(f"  Best params : {gs.best_params_}")
    print(f"  CV F1-macro : {gs.best_score_:.4f}")

    # report dạng dict thay vì in text thô
    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True
    )

    # Gom toàn bộ report của model thành 1 hàng
    flat_row = {"model": name}

    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):  # các class, macro avg, weighted avg
            for metric_name, value in metrics.items():
                flat_row[f"{label}_{metric_name}"] = value
        else:  # accuracy là scalar
            flat_row[label] = metrics

    flat_row["best_params"] = str(gs.best_params_)
    flat_row["cv_f1_macro"] = gs.best_score_
    report_rows.append(flat_row)

    # phần cũ của bạn nếu vẫn muốn giữ
    row = compute_metrics(y_test, y_pred, name)
    row["best_params"] = str(gs.best_params_)
    all_results.append(row)

# Bảng classification report: mỗi hàng là 1 model
classification_report_df = pd.DataFrame(report_rows)

# Sắp xếp cột cho dễ nhìn
first_cols = ["model", "accuracy", "cv_f1_macro", "best_params"]
other_cols = [c for c in classification_report_df.columns if c not in first_cols]
classification_report_df = classification_report_df[first_cols + other_cols]

display(classification_report_df)

pd.DataFrame(all_results).to_csv(
    output_path("all_results_models.csv"),
    index=False,
    encoding="utf-8-sig"
)

N_CLASSES : 8
CLASS_NAMES: ['0', '1', '2', '3', '4', '5', '6', '7']

[KNN] GridSearch đang chạy...
  Best params : {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
  CV F1-macro : 0.5843

[DecisionTree] GridSearch đang chạy...
  Best params : {'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 10}
  CV F1-macro : 0.6239

[RandomForest] GridSearch đang chạy...
  Best params : {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
  CV F1-macro : 0.7796

[GaussianNB] GridSearch đang chạy...
  Best params : {'var_smoothing': 1e-09}
  CV F1-macro : 0.3770

[LogisticRegression] GridSearch đang chạy...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

  Best params : {'C': 0.01, 'multi_class': 'multinomial', 'solver': 'lbfgs'}
  CV F1-macro : 0.7890

[LinearSVM] GridSearch đang chạy...
  Best params : {'C': 0.01}
  CV F1-macro : 0.7837

[RBF_SVM] GridSearch đang chạy...
  Best params : {'C': 10, 'gamma': 'auto'}
  CV F1-macro : 0.8535


,model,accuracy,cv_f1_macro,best_params,0_precision,0_recall,0_f1-score,0_support,1_precision,1_recall,...,7_f1-score,7_support,macro avg_precision,macro avg_recall,macro avg_f1-score,macro avg_support,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,weighted avg_support
0,KNN,0.6095,0.584337,"{'metric': 'manhattan', 'n_neighbors': 11, 'we...",0.561224,0.220,0.316092,250.0,0.459514,0.908,...,0.714932,250.0,0.637660,0.6095,0.598253,2000.0,0.637660,0.6095,0.598253,2000.0
1,DecisionTree,0.6480,0.623946,"{'criterion': 'entropy', 'max_depth': 10, 'min...",0.558824,0.456,0.502203,250.0,0.728625,0.784,...,0.621094,250.0,0.652306,0.6480,0.646255,2000.0,0.652306,0.6480,0.646255,2000.0
2,RandomForest,0.7950,0.779646,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.815029,0.564,0.666667,250.0,0.824176,0.900,...,0.783550,250.0,0.804108,0.7950,0.793114,2000.0,0.804108,0.7950,0.793114,2000.0
3,GaussianNB,0.3990,0.376994,{'var_smoothing': 1e-09},0.342466,0.100,0.154799,250.0,0.407336,0.844,...,0.519084,250.0,0.432931,0.3990,0.383486,2000.0,0.432931,0.3990,0.383486,2000.0
4,LogisticRegression,0.7830,0.788984,"{'C': 0.01, 'multi_class': 'multinomial', 'sol...",0.696035,0.632,0.662474,250.0,0.835165,0.912,...,0.778226,250.0,0.782041,0.7830,0.782082,2000.0,0.782041,0.7830,0.782082,2000.0
5,LinearSVM,0.7855,0.783660,{'C': 0.01},0.700000,0.588,0.639130,250.0,0.834532,0.928,...,0.784553,250.0,0.783493,0.7855,0.783495,2000.0,0.783493,0.7855,0.783495,2000.0
6,RBF_SVM,0.8515,0.853505,"{'C': 10, 'gamma': 'auto'}",0.755102,0.740,0.747475,250.0,0.865385,0.900,...,0.872211,250.0,0.851481,0.8515,0.851264,2000.0,0.851481,0.8515,0.851264,2000.0


In [10]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC


# =========================
# 1) THÔNG TIN NHÃN
# =========================
unique_classes = np.unique(y_train)
N_CLASSES = len(unique_classes)
CLASS_NAMES = [str(c) for c in unique_classes]

print("N_CLASSES :", N_CLASSES)
print("CLASS_NAMES:", CLASS_NAMES)


# =========================
# 2) HÀM TÍNH METRICS
# =========================
def compute_metrics(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


# =========================
# 3) CÁC MÔ HÌNH VỚI BEST PARAMS
# =========================
BEST_MODELS = [
    (
        "KNN",
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(
                metric="manhattan",
                n_neighbors=11,
                weights="distance"
            ))
        ])
    ),
    (
        "DecisionTree",
        DecisionTreeClassifier(
            criterion="entropy",
            max_depth=10,
            min_samples_split=2,
            random_state=42
        )
    ),
    (
        "RandomForest",
        RandomForestClassifier(
            max_depth=None,
            min_samples_split=2,
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    ),
    (
        "GaussianNB",
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", GaussianNB(var_smoothing=1e-6))
        ])
    ),
    (
        "LogisticRegression",
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=0.01,
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=1000,
                random_state=42
            ))
        ])
    ),
    (
        "LinearSVM",
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                C=0.01,
                max_iter=2000,
                random_state=42
            ))
        ])
    ),
    (
        "RBF_SVM",
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                C=10,
                gamma="auto",
                kernel="rbf",
                probability=True,
                random_state=42
            ))
        ])
    ),
]


# =========================
# 4) TRAIN TỪNG MODEL
# =========================
all_results = []
report_rows = []
fitted_models = {}

for name, model in BEST_MODELS:
    print(f"\n[{name}] Đang train...")

    model.fit(X_train_selected, y_train)
    fitted_models[name] = model

    y_pred = model.predict(X_test_selected)

    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True
    )

    flat_row = {"model": name}

    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):
            for metric_name, value in metrics.items():
                flat_row[f"{label}_{metric_name}"] = value
        else:
            flat_row[label] = metrics

    flat_row["best_params"] = "fixed_best_params"
    flat_row["cv_f1_macro"] = np.nan
    report_rows.append(flat_row)

    row = compute_metrics(y_test, y_pred, name)
    row["best_params"] = "fixed_best_params"
    row["cv_f1_macro"] = np.nan
    all_results.append(row)


# =========================
# 5) STACKING MODEL
# =========================
stack_estimators = [
    ("knn", clone(fitted_models["KNN"])),
    ("dt", clone(fitted_models["DecisionTree"])),
    ("rf", clone(fitted_models["RandomForest"])),
    ("gnb", clone(fitted_models["GaussianNB"])),
    ("lr", clone(fitted_models["LogisticRegression"])),
    ("lsvm", clone(fitted_models["LinearSVM"])),
    ("rbfsvm", clone(fitted_models["RBF_SVM"]))
]

stack_model = StackingClassifier(
    estimators=stack_estimators,
    final_estimator=LogisticRegression(max_iter=2000, random_state=42),
    cv=5,
    stack_method="auto",
    n_jobs=-1,
    passthrough=False
)

print("\n[StackingClassifier] Đang train...")

stack_model.fit(X_train_selected, y_train)
y_pred_stack = stack_model.predict(X_test_selected)

stack_report_dict = classification_report(
    y_test,
    y_pred_stack,
    target_names=CLASS_NAMES,
    zero_division=0,
    output_dict=True
)

stack_flat_row = {"model": "StackingClassifier"}
for label, metrics in stack_report_dict.items():
    if isinstance(metrics, dict):
        for metric_name, value in metrics.items():
            stack_flat_row[f"{label}_{metric_name}"] = value
    else:
        stack_flat_row[label] = metrics

stack_flat_row["best_params"] = "base_models_fixed_best_params"
stack_flat_row["cv_f1_macro"] = np.nan
report_rows.append(stack_flat_row)

stack_row = compute_metrics(y_test, y_pred_stack, "StackingClassifier")
stack_row["best_params"] = "base_models_fixed_best_params"
stack_row["cv_f1_macro"] = np.nan
all_results.append(stack_row)


# =========================
# 6) DATAFRAME KẾT QUẢ
# =========================
classification_report_df = pd.DataFrame(report_rows)

first_cols = ["model", "accuracy", "cv_f1_macro", "best_params"]
other_cols = [c for c in classification_report_df.columns if c not in first_cols]
classification_report_df = classification_report_df[first_cols + other_cols]

results_df = pd.DataFrame(all_results)

display(classification_report_df)
display(results_df)


# =========================
# 7) LƯU FILE CSV
# =========================
classification_report_df.to_csv(
    output_path("classification_report_all_models_and_stacking.csv"),
    index=False,
    encoding="utf-8-sig"
)

results_df.to_csv(
    output_path("all_results_models_and_stacking.csv"),
    index=False,
    encoding="utf-8-sig"
)

N_CLASSES : 8
CLASS_NAMES: ['0', '1', '2', '3', '4', '5', '6', '7']

[KNN] Đang train...

[DecisionTree] Đang train...

[RandomForest] Đang train...

[GaussianNB] Đang train...

[LogisticRegression] Đang train...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



[LinearSVM] Đang train...

[RBF_SVM] Đang train...

[StackingClassifier] Đang train...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

,model,accuracy,cv_f1_macro,best_params,0_precision,0_recall,0_f1-score,0_support,1_precision,1_recall,...,7_f1-score,7_support,macro avg_precision,macro avg_recall,macro avg_f1-score,macro avg_support,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,weighted avg_support
0,KNN,0.5270,NaN,fixed_best_params,0.475610,0.156,0.234940,250.0,0.396679,0.860,...,0.611765,250.0,0.567279,0.5270,0.515808,2000.0,0.567279,0.5270,0.515808,2000.0
1,DecisionTree,0.6480,NaN,fixed_best_params,0.550926,0.476,0.510730,250.0,0.733083,0.780,...,0.617188,250.0,0.652516,0.6480,0.646669,2000.0,0.652516,0.6480,0.646669,2000.0
2,RandomForest,0.7965,NaN,fixed_best_params,0.811429,0.568,0.668235,250.0,0.803571,0.900,...,0.790598,250.0,0.805365,0.7965,0.794813,2000.0,0.805365,0.7965,0.794813,2000.0
3,GaussianNB,0.3985,NaN,fixed_best_params,0.337500,0.108,0.163636,250.0,0.407336,0.844,...,0.519084,250.0,0.432142,0.3985,0.384166,2000.0,0.432142,0.3985,0.384166,2000.0
4,LogisticRegression,0.7815,NaN,fixed_best_params,0.704545,0.620,0.659574,250.0,0.816547,0.908,...,0.773931,250.0,0.780276,0.7815,0.780168,2000.0,0.780276,0.7815,0.780168,2000.0
5,LinearSVM,0.7840,NaN,fixed_best_params,0.728643,0.580,0.645880,250.0,0.816901,0.928,...,0.779798,250.0,0.782008,0.7840,0.781381,2000.0,0.782008,0.7840,0.781381,2000.0
6,RBF_SVM,0.8220,NaN,fixed_best_params,0.695122,0.684,0.689516,250.0,0.872587,0.904,...,0.847251,250.0,0.822828,0.8220,0.822042,2000.0,0.822828,0.8220,0.822042,2000.0
7,StackingClassifier,0.8605,NaN,base_models_fixed_best_params,0.785714,0.748,0.766393,250.0,0.901961,0.920,...,0.883534,250.0,0.860535,0.8605,0.860338,2000.0,0.860535,0.8605,0.860338,2000.0


,model,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_params,cv_f1_macro
0,KNN,0.5270,0.567279,0.5270,0.515808,0.567279,0.5270,0.515808,fixed_best_params,NaN
1,DecisionTree,0.6480,0.652516,0.6480,0.646669,0.652516,0.6480,0.646669,fixed_best_params,NaN
2,RandomForest,0.7965,0.805365,0.7965,0.794813,0.805365,0.7965,0.794813,fixed_best_params,NaN
3,GaussianNB,0.3985,0.432142,0.3985,0.384166,0.432142,0.3985,0.384166,fixed_best_params,NaN
4,LogisticRegression,0.7815,0.780276,0.7815,0.780168,0.780276,0.7815,0.780168,fixed_best_params,NaN
5,LinearSVM,0.7840,0.782008,0.7840,0.781381,0.782008,0.7840,0.781381,fixed_best_params,NaN
6,RBF_SVM,0.8220,0.822828,0.8220,0.822042,0.822828,0.8220,0.822042,fixed_best_params,NaN
7,StackingClassifier,0.8605,0.860535,0.8605,0.860338,0.860535,0.8605,0.860338,base_models_fixed_best_params,NaN


In [11]:
!pip install causal-conv1d>=1.4.0 --no-build-isolation
!pip install mamba-ssm --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl size=533592144 sha256=d66a3c4c94a693e02d341cace7a6af0b72177b6afa655a25e3a6505130a68cbf
  Stored in directory: /root/.cache/pip/wheels/28/83/54/d45107838fec575b93f5d723f56351cee19a1b13bcd4ec9f3f
Successfully built mamba-ssm


In [12]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

# official package
from mamba_ssm import Mamba


class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(np.asarray(X), dtype=torch.float32)
        self.y = torch.tensor(np.asarray(y), dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = Mamba(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, L, D)
        residual = x
        x = self.norm(x)
        x = self.mamba(x)
        x = self.dropout(x)
        return x + residual


class MambaTabularClassifier(nn.Module):
    def __init__(
        self,
        n_features,
        n_classes,
        d_model=64,
        n_layers=3,
        d_state=16,
        d_conv=4,
        expand=2,
        dropout=0.1,
        pooling="mean"
    ):
        super().__init__()
        self.n_features = n_features
        self.n_classes = n_classes
        self.pooling = pooling

        # mỗi feature là 1 token scalar -> chiếu lên d_model
        self.input_proj = nn.Linear(1, d_model)

        # learnable positional embedding theo số feature
        self.pos_embed = nn.Parameter(torch.zeros(1, n_features, d_model))

        self.blocks = nn.ModuleList([
            MambaBlock(
                d_model=d_model,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, x):
        # x: (B, F)
        x = x.unsqueeze(-1)                 # (B, F, 1)
        x = self.input_proj(x)             # (B, F, D)
        x = x + self.pos_embed

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        if self.pooling == "mean":
            x = x.mean(dim=1)              # (B, D)
        elif self.pooling == "max":
            x = x.max(dim=1).values
        elif self.pooling == "last":
            x = x[:, -1, :]
        else:
            raise ValueError("pooling must be one of: mean, max, last")

        logits = self.head(x)
        return logits

In [13]:
def compute_metrics(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


@torch.no_grad()
def evaluate_model(model, loader, device, class_names):
    model.eval()

    all_preds = []
    all_targets = []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        preds = torch.argmax(logits, dim=1)

        total_loss += loss.item() * xb.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(yb.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    report_dict = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        zero_division=0,
        output_dict=True
    )

    return avg_loss, np.array(all_targets), np.array(all_preds), report_dict


def train_mamba_model(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    n_classes,
    class_names,
    batch_size=64,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=100,
    d_model=64,
    n_layers=3,
    d_state=16,
    d_conv=4,
    expand=2,
    dropout=0.1,
    pooling="mean",
    patience=8,
    device=None
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    train_ds = TabularDataset(X_train, y_train)
    val_ds   = TabularDataset(X_val, y_val)
    test_ds  = TabularDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    model = MambaTabularClassifier(
        n_features=X_train.shape[1],
        n_classes=n_classes,
        d_model=d_model,
        n_layers=n_layers,
        d_state=d_state,
        d_conv=d_conv,
        expand=expand,
        dropout=dropout,
        pooling=pooling
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3
    )

    best_val_f1 = -1.0
    best_state = None
    best_epoch = 0
    wait = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        val_loss, y_val_true, y_val_pred, _ = evaluate_model(
            model, val_loader, device, class_names
        )
        val_f1 = f1_score(y_val_true, y_val_pred, average="macro", zero_division=0)
        val_acc = accuracy_score(y_val_true, y_val_pred)

        scheduler.step(val_f1)

        print(
            f"Epoch {epoch:02d} | "
            f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val F1: {val_f1:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            wait = 0
            print(f"  --> Best model updated at epoch {epoch}")
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)

    test_loss, y_test_true, y_test_pred, test_report = evaluate_model(
        model, test_loader, device, class_names
    )

    metrics_row = compute_metrics(y_test_true, y_test_pred, "Mamba")
    metrics_row["best_epoch"] = best_epoch
    metrics_row["best_val_f1_macro"] = best_val_f1

    flat_row = {"model": "Mamba"}
    for label, metrics in test_report.items():
        if isinstance(metrics, dict):
            for metric_name, value in metrics.items():
                flat_row[f"{label}_{metric_name}"] = value
        else:
            flat_row[label] = metrics

    flat_row["best_epoch"] = best_epoch
    flat_row["best_val_f1_macro"] = best_val_f1
    flat_row["test_loss"] = test_loss

    return model, metrics_row, flat_row, (y_test_true, y_test_pred)

In [14]:
unique_classes = np.unique(y_train)
N_CLASSES = len(unique_classes)
CLASS_NAMES = [str(c) for c in unique_classes]

model_mamba, mamba_metrics_row, mamba_report_row, test_outputs = train_mamba_model(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test,
    n_classes=N_CLASSES,
    class_names=CLASS_NAMES,
    batch_size=64,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=30,
    d_model=64,
    n_layers=3,
    d_state=16,
    d_conv=4,
    expand=2,
    dropout=0.1,
    pooling="mean",
    patience=8
)

print("\nMamba metrics:")
print(mamba_metrics_row)

mamba_report_df = pd.DataFrame([mamba_report_row])
mamba_results_df = pd.DataFrame([mamba_metrics_row])

display(mamba_report_df)
display(mamba_results_df)

Epoch 01 | LR: 0.001000 | Train Loss: 1.9485 | Val Loss: 1.8502 | Val Acc: 0.2545 | Val F1: 0.1777
  --> Best model updated at epoch 1
Epoch 02 | LR: 0.001000 | Train Loss: 1.8016 | Val Loss: 1.7661 | Val Acc: 0.2820 | Val F1: 0.1972
  --> Best model updated at epoch 2
Epoch 03 | LR: 0.001000 | Train Loss: 1.6902 | Val Loss: 1.7194 | Val Acc: 0.2945 | Val F1: 0.2576
  --> Best model updated at epoch 3
Epoch 04 | LR: 0.001000 | Train Loss: 1.6122 | Val Loss: 1.6015 | Val Acc: 0.3605 | Val F1: 0.3105
  --> Best model updated at epoch 4
Epoch 05 | LR: 0.001000 | Train Loss: 1.5480 | Val Loss: 1.5574 | Val Acc: 0.3830 | Val F1: 0.3588
  --> Best model updated at epoch 5
Epoch 06 | LR: 0.001000 | Train Loss: 1.4139 | Val Loss: 1.3296 | Val Acc: 0.4885 | Val F1: 0.4744
  --> Best model updated at epoch 6
Epoch 07 | LR: 0.001000 | Train Loss: 1.1885 | Val Loss: 1.1002 | Val Acc: 0.5775 | Val F1: 0.5568
  --> Best model updated at epoch 7
Epoch 08 | LR: 0.001000 | Train Loss: 1.0202 | Val Loss

,model,0_precision,0_recall,0_f1-score,0_support,1_precision,1_recall,1_f1-score,1_support,2_precision,...,macro avg_recall,macro avg_f1-score,macro avg_support,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,weighted avg_support,best_epoch,best_val_f1_macro,test_loss
0,Mamba,0.676329,0.56,0.612691,250.0,0.925439,0.844,0.882845,250.0,0.796226,...,0.8005,0.798549,2000.0,0.80012,0.8005,0.798549,2000.0,26,0.796321,0.620296


,model,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,Mamba,0.8005,0.80012,0.8005,0.798549,0.80012,0.8005,0.798549,26,0.796321


In [15]:
import pandas as pd
import numpy as np
import torch
import gc

candidate_configs = [
    {"lr": 1e-3, "d_model": 64, "n_layers": 3, "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.2, "weight_decay": 1e-4, "batch_size": 64, "pooling": "mean"},
    {"lr": 1e-3, "d_model": 64, "n_layers": 3, "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.3, "weight_decay": 1e-4, "batch_size": 64, "pooling": "mean"},
    {"lr": 3e-4, "d_model": 64, "n_layers": 3, "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.2, "weight_decay": 1e-4, "batch_size": 64, "pooling": "mean"},
    {"lr": 1e-3, "d_model": 32, "n_layers": 3, "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.2, "weight_decay": 1e-4, "batch_size": 64, "pooling": "mean"},
    {"lr": 1e-3, "d_model": 64, "n_layers": 2, "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.2, "weight_decay": 1e-3, "batch_size": 64, "pooling": "mean"},
    {"lr": 1e-3, "d_model": 64, "n_layers": 3, "d_state": 32, "d_conv": 4, "expand": 2, "dropout": 0.2, "weight_decay": 1e-4, "batch_size": 64, "pooling": "mean"},
]

unique_classes = np.unique(y_train)
N_CLASSES = len(unique_classes)
CLASS_NAMES = [str(c) for c in unique_classes]

search_results = []
search_report_rows = []
best_model = None
best_config = None
best_score = -1.0

for idx, cfg in enumerate(candidate_configs, 1):
    print("\n" + "=" * 80)
    print(f"Trial {idx}/{len(candidate_configs)}")
    print(cfg)
    print("=" * 80)

    model, metrics_row, report_row, _ = train_mamba_model(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
        n_classes=N_CLASSES,
        class_names=CLASS_NAMES,
        batch_size=cfg["batch_size"],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        epochs=100,
        d_model=cfg["d_model"],
        n_layers=cfg["n_layers"],
        d_state=cfg["d_state"],
        d_conv=cfg["d_conv"],
        expand=cfg["expand"],
        dropout=cfg["dropout"],
        pooling=cfg["pooling"],
        patience=8
    )

    row = {
        "trial": idx,
        **cfg,
        "test_accuracy": metrics_row["accuracy"],
        "test_precision_macro": metrics_row["precision_macro"],
        "test_recall_macro": metrics_row["recall_macro"],
        "test_f1_macro": metrics_row["f1_macro"],
        "test_precision_weighted": metrics_row["precision_weighted"],
        "test_recall_weighted": metrics_row["recall_weighted"],
        "test_f1_weighted": metrics_row["f1_weighted"],
        "best_epoch": metrics_row["best_epoch"],
        "best_val_f1_macro": metrics_row["best_val_f1_macro"],
    }
    search_results.append(row)

    flat_report_row = {
        "trial": idx,
        "model": f"Mamba_trial_{idx}",
        **cfg,
        **report_row
    }
    search_report_rows.append(flat_report_row)

    if metrics_row["best_val_f1_macro"] > best_score:
        best_score = metrics_row["best_val_f1_macro"]
        best_config = cfg.copy()
        best_model = model

    # dọn bộ nhớ GPU giữa các trial
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

search_df = pd.DataFrame(search_results)
search_df = search_df.sort_values(
    by=["best_val_f1_macro", "test_f1_macro", "test_accuracy"],
    ascending=False
).reset_index(drop=True)

search_report_df = pd.DataFrame(search_report_rows)

print("\nTop cấu hình tốt nhất:")
display(search_df)

print("\nBest config:")
print(best_config)
print(f"Best validation F1-macro: {best_score:.6f}")


Trial 1/6
{'lr': 0.001, 'd_model': 64, 'n_layers': 3, 'd_state': 16, 'd_conv': 4, 'expand': 2, 'dropout': 0.2, 'weight_decay': 0.0001, 'batch_size': 64, 'pooling': 'mean'}
Epoch 01 | LR: 0.001000 | Train Loss: 1.9506 | Val Loss: 1.8369 | Val Acc: 0.2530 | Val F1: 0.1543
  --> Best model updated at epoch 1
Epoch 02 | LR: 0.001000 | Train Loss: 1.7736 | Val Loss: 1.7557 | Val Acc: 0.2715 | Val F1: 0.1919
  --> Best model updated at epoch 2
Epoch 03 | LR: 0.001000 | Train Loss: 1.6459 | Val Loss: 1.6208 | Val Acc: 0.3780 | Val F1: 0.3263
  --> Best model updated at epoch 3
Epoch 04 | LR: 0.001000 | Train Loss: 1.4747 | Val Loss: 1.2798 | Val Acc: 0.5215 | Val F1: 0.4941
  --> Best model updated at epoch 4
Epoch 05 | LR: 0.001000 | Train Loss: 1.2592 | Val Loss: 1.2263 | Val Acc: 0.5290 | Val F1: 0.4995
  --> Best model updated at epoch 5
Epoch 06 | LR: 0.001000 | Train Loss: 1.1447 | Val Loss: 1.0730 | Val Acc: 0.5860 | Val F1: 0.5700
  --> Best model updated at epoch 6
Epoch 07 | LR: 0.

,trial,lr,d_model,n_layers,d_state,d_conv,expand,dropout,weight_decay,batch_size,pooling,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_precision_weighted,test_recall_weighted,test_f1_weighted,best_epoch,best_val_f1_macro
0,1,0.0010,64,3,16,4,2,0.2,0.0001,64,mean,0.8190,0.817608,0.8190,0.817246,0.817608,0.8190,0.817246,51,0.828947
1,5,0.0010,64,2,16,4,2,0.2,0.0010,64,mean,0.8160,0.815668,0.8160,0.814791,0.815668,0.8160,0.814791,42,0.805930
2,2,0.0010,64,3,16,4,2,0.3,0.0001,64,mean,0.7750,0.779592,0.7750,0.774497,0.779592,0.7750,0.774497,42,0.788640
3,4,0.0010,32,3,16,4,2,0.2,0.0001,64,mean,0.7830,0.783574,0.7830,0.782463,0.783574,0.7830,0.782463,51,0.787699
4,6,0.0010,64,3,32,4,2,0.2,0.0001,64,mean,0.7805,0.779413,0.7805,0.777317,0.779413,0.7805,0.777317,27,0.787046
5,3,0.0003,64,3,16,4,2,0.2,0.0001,64,mean,0.7365,0.733854,0.7365,0.731916,0.733854,0.7365,0.731916,78,0.741714



Best config:
{'lr': 0.001, 'd_model': 64, 'n_layers': 3, 'd_state': 16, 'd_conv': 4, 'expand': 2, 'dropout': 0.2, 'weight_decay': 0.0001, 'batch_size': 64, 'pooling': 'mean'}
Best validation F1-macro: 0.828947


In [16]:
!pip install causal-conv1d>=1.4.0 --no-build-isolation
!pip install mamba-ssm --no-build-isolation

In [17]:
import pandas as pd

# Đọc dữ liệu
X_train = pd.read_csv(resolve_data_path("X_train.csv"))
X_val = pd.read_csv(resolve_data_path("X_val.csv"))
X_test = pd.read_csv(resolve_data_path("X_test.csv"))

y_train = pd.read_csv(resolve_data_path("y_train.csv")).iloc[:, 0].values
y_val = pd.read_csv(resolve_data_path("y_val.csv")).iloc[:, 0].values
y_test = pd.read_csv(resolve_data_path("y_test.csv")).iloc[:, 0].values

In [18]:
config = {
    "DEVICE": "cuda",
    "NUM_WORKERS": 2,
    "SEED": 42,
    "DEFAULT_BATCH_SIZE": 64,
    "DEFAULT_EPOCHS": 50,
    "DEFAULT_PATIENCE": 8,
}

import torch
import torch.nn as nn
from mamba_ssm import Mamba


class MambaTab(nn.Module):
    """
    Mamba for tabular classification.

    Input:
        x: [B, F]
    Output:
        logits: [B, C]
    """
    def __init__(
        self,
        input_features: int,
        n_class: int,
        d_model: int = 64,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
        dropout: float = 0.2,
        pooling: str = "mean",
    ):
        super().__init__()

        self.input_features = input_features
        self.n_class = n_class
        self.d_model = d_model
        self.pooling = pooling

        # Mỗi feature scalar -> embedding vector
        self.feature_embed = nn.Linear(1, d_model)

        self.norm = nn.LayerNorm(d_model)
        self.act = nn.ReLU()

        self.mamba = Mamba(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand
        )

        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_class)

    def forward(self, x):
        # x: [B, F]
        x = x.unsqueeze(-1)          # [B, F, 1]
        x = self.feature_embed(x)    # [B, F, d_model]
        x = self.norm(x)
        x = self.act(x)

        x = self.mamba(x)            # [B, F, d_model]

        if self.pooling == "mean":
            x = x.mean(dim=1)
        elif self.pooling == "max":
            x = x.max(dim=1).values
        elif self.pooling == "last":
            x = x[:, -1, :]
        else:
            raise ValueError(f"Unsupported pooling: {self.pooling}")

        x = self.dropout(x)
        logits = self.head(x)
        return logits

In [19]:
import os
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


# =========================================================
# 1. Reproducibility
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =========================================================
# 2. Dataset
# =========================================================
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(np.asarray(X), dtype=torch.float32)
        self.y = torch.tensor(np.asarray(y), dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# =========================================================
# 3. DataLoader helper
# =========================================================
def build_loaders(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    batch_size=64,
    num_workers=2
):
    train_ds = TabularDataset(X_train, y_train)
    val_ds = TabularDataset(X_val, y_val)
    test_ds = TabularDataset(X_test, y_test)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader


# =========================================================
# 4. Scaling helper
# =========================================================
def scale_data(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler


# =========================================================
# 5. Train / eval
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1, np.array(all_labels), np.array(all_preds)


# =========================================================
# 6. Fit one config
# =========================================================
def fit_one_config(
    model,
    train_loader,
    val_loader,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=50,
    patience=8,
    use_class_weights=False,
    y_train=None,
    save_path="best_model.pt"
):
    if use_class_weights:
        class_counts = np.bincount(y_train)
        class_weights = len(y_train) / (len(class_counts) * class_counts)
        class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )

    best_val_f1 = -1.0
    best_state = None
    wait = 0

    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc, val_f1, _, _ = evaluate(
            model, val_loader, criterion, device
        )

        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)

        history.append({
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
        })

        print(
            f"Epoch {epoch:03d} | "
            f"LR: {current_lr:.6f} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, save_path)
            wait = 0
            print(f"  --> Best model updated at epoch {epoch}")
        else:
            wait += 1

        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_f1


# =========================================================
# 7. Hyperparameter search
# =========================================================
def hyperparameter_search(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    input_features,
    n_classes,
    class_names=None,
    candidate_configs=None,
    device="cuda",
    num_workers=2,
    output_dir=str(RESULT_DIR / "mamba_search_results"),
    use_class_weights=False
):
    os.makedirs(output_dir, exist_ok=True)

    results = []
    best_model_path = None
    best_config = None
    best_val_f1 = -1.0
    best_history = None

    if candidate_configs is None:
        candidate_configs = [
            {
                "lr": 1e-3, "d_model": 32, "d_state": 16, "d_conv": 4, "expand": 2,
                "dropout": 0.2, "weight_decay": 1e-4, "batch_size": 64,
                "pooling": "mean", "epochs": 40, "patience": 8
            },
            {
                "lr": 1e-3, "d_model": 64, "d_state": 16, "d_conv": 4, "expand": 2,
                "dropout": 0.2, "weight_decay": 1e-4, "batch_size": 64,
                "pooling": "mean", "epochs": 40, "patience": 8
            },
            {
                "lr": 3e-4, "d_model": 64, "d_state": 16, "d_conv": 4, "expand": 2,
                "dropout": 0.3, "weight_decay": 1e-4, "batch_size": 64,
                "pooling": "mean", "epochs": 50, "patience": 10
            },
            {
                "lr": 3e-4, "d_model": 128, "d_state": 16, "d_conv": 4, "expand": 2,
                "dropout": 0.3, "weight_decay": 1e-4, "batch_size": 32,
                "pooling": "mean", "epochs": 50, "patience": 10
            },
            {
                "lr": 1e-4, "d_model": 64, "d_state": 32, "d_conv": 4, "expand": 2,
                "dropout": 0.3, "weight_decay": 5e-4, "batch_size": 64,
                "pooling": "mean", "epochs": 60, "patience": 10
            },
            {
                "lr": 3e-4, "d_model": 64, "d_state": 16, "d_conv": 2, "expand": 2,
                "dropout": 0.2, "weight_decay": 1e-5, "batch_size": 64,
                "pooling": "max", "epochs": 50, "patience": 10
            },
        ]

    for idx, hp in enumerate(candidate_configs, start=1):
        print("\n" + "=" * 80)
        print(f"[Config {idx}/{len(candidate_configs)}] {hp}")
        print("=" * 80)

        train_loader, val_loader, test_loader = build_loaders(
            X_train, y_train,
            X_val, y_val,
            X_test, y_test,
            batch_size=hp["batch_size"],
            num_workers=num_workers
        )

        model = MambaTab(
            input_features=input_features,
            n_class=n_classes,
            d_model=hp["d_model"],
            d_state=hp["d_state"],
            d_conv=hp["d_conv"],
            expand=hp["expand"],
            dropout=hp["dropout"],
            pooling=hp["pooling"]
        ).to(device)

        save_path = os.path.join(output_dir, f"best_model_config_{idx}.pt")

        model, history_df, current_best_val_f1 = fit_one_config(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            lr=hp["lr"],
            weight_decay=hp["weight_decay"],
            epochs=hp["epochs"],
            patience=hp["patience"],
            use_class_weights=use_class_weights,
            y_train=y_train,
            save_path=save_path
        )

        criterion = nn.CrossEntropyLoss()
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        test_loss, test_acc, test_f1, _, _ = evaluate(model, test_loader, criterion, device)

        row = {
            "config_id": idx,
            **hp,
            "best_val_f1": val_f1,
            "val_acc": val_acc,
            "val_loss": val_loss,
            "test_acc": test_acc,
            "test_f1": test_f1,
            "test_loss": test_loss,
        }
        results.append(row)

        history_df.to_csv(
            os.path.join(output_dir, f"history_config_{idx}.csv"),
            index=False
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_config = hp
            best_model_path = save_path
            best_history = history_df.copy()

    results_df = pd.DataFrame(results).sort_values(
        by=["best_val_f1", "test_f1"],
        ascending=False
    ).reset_index(drop=True)

    results_df.to_csv(os.path.join(output_dir, "hyperparameter_results.csv"), index=False)

    print("\n===== ALL SEARCH RESULTS =====")
    print(results_df)

    print("\n===== BEST CONFIG =====")
    print(best_config)
    print(f"Best Val F1: {best_val_f1:.4f}")
    print(f"Best model path: {best_model_path}")

    # Load lại best model
    best_model = MambaTab(
        input_features=input_features,
        n_class=n_classes,
        d_model=best_config["d_model"],
        d_state=best_config["d_state"],
        d_conv=best_config["d_conv"],
        expand=best_config["expand"],
        dropout=best_config["dropout"],
        pooling=best_config["pooling"]
    ).to(device)

    best_model.load_state_dict(torch.load(best_model_path, map_location=device))

    return best_model, best_config, results_df, best_history


# =========================================================
# 8. Final test
# =========================================================
def test_model(model, test_loader, device, class_names=None):
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, test_f1, y_true, y_pred = evaluate(
        model, test_loader, criterion, device
    )

    print("\n===== FINAL TEST =====")
    print(f"Test Loss : {test_loss:.4f}")
    print(f"Test Acc  : {test_acc:.4f}")
    print(f"Test F1   : {test_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_true, y_pred,
        target_names=class_names,
        zero_division=0
    ))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return {
        "test_loss": test_loss,
        "test_acc": test_acc,
        "test_f1": test_f1,
        "y_true": y_true,
        "y_pred": y_pred
    }


# =========================================================
# 9. Example usage
# =========================================================
if __name__ == "__main__":
    set_seed(config["SEED"])

    device = torch.device(
        config["DEVICE"] if torch.cuda.is_available() else "cpu"
    )
    print("Using device:", device)

    # =====================================================
    # THAY DỮ LIỆU Ở ĐÂY BẰNG DỮ LIỆU THẬT CỦA BẠN
    # X_train, X_val, X_test, y_train, y_val, y_test
    # =====================================================
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split

    X_train, X_val, X_test, scaler = scale_data(X_train, X_val, X_test)

    input_features = X_train.shape[1]
    n_classes = len(np.unique(y_train))
    class_names = [str(c) for c in sorted(np.unique(y_train))]

    candidate_configs = [
        {"lr": 1e-3, "d_model": 32,  "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.2, "weight_decay": 1e-4, "batch_size": 64, "pooling": "mean", "epochs": 40, "patience": 8},
        {"lr": 1e-3, "d_model": 64,  "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.2, "weight_decay": 1e-4, "batch_size": 64, "pooling": "mean", "epochs": 40, "patience": 8},
        {"lr": 3e-4, "d_model": 64,  "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.3, "weight_decay": 1e-4, "batch_size": 64, "pooling": "mean", "epochs": 50, "patience": 10},
        {"lr": 3e-4, "d_model": 128, "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.3, "weight_decay": 1e-4, "batch_size": 32, "pooling": "mean", "epochs": 50, "patience": 10},
        {"lr": 1e-4, "d_model": 64,  "d_state": 32, "d_conv": 4, "expand": 2, "dropout": 0.3, "weight_decay": 5e-4, "batch_size": 64, "pooling": "mean", "epochs": 60, "patience": 10},
        {"lr": 3e-4, "d_model": 64,  "d_state": 16, "d_conv": 2, "expand": 2, "dropout": 0.2, "weight_decay": 1e-5, "batch_size": 64, "pooling": "max",  "epochs": 50, "patience": 10},
    ]

    best_model, best_config, results_df, best_history = hyperparameter_search(
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        X_test=X_test, y_test=y_test,
        input_features=input_features,
        n_classes=n_classes,
        class_names=class_names,
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        output_dir=str(RESULT_DIR / "mamba_search_results")
    )

    # test loader cho best model
    _, _, test_loader = build_loaders(
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        batch_size=best_config["batch_size"],
        num_workers=config["NUM_WORKERS"]
    )

    final_results = test_model(
        model=best_model,
        test_loader=test_loader,
        device=device,
        class_names=class_names
    )

Using device: cuda

[Config 1/6] {'lr': 0.001, 'd_model': 32, 'd_state': 16, 'd_conv': 4, 'expand': 2, 'dropout': 0.2, 'weight_decay': 0.0001, 'batch_size': 64, 'pooling': 'mean', 'epochs': 40, 'patience': 8}
Epoch 001 | LR: 0.001000 | Train Loss: 1.9939 | Train Acc: 0.1824 | Train F1: 0.1579 | Val Loss: 1.9327 | Val Acc: 0.1950 | Val F1: 0.1448
  --> Best model updated at epoch 1
Epoch 002 | LR: 0.001000 | Train Loss: 1.8921 | Train Acc: 0.2277 | Train F1: 0.2068 | Val Loss: 1.9059 | Val Acc: 0.2450 | Val F1: 0.2009
  --> Best model updated at epoch 2
Epoch 003 | LR: 0.001000 | Train Loss: 1.8809 | Train Acc: 0.2390 | Train F1: 0.2059 | Val Loss: 1.9047 | Val Acc: 0.2400 | Val F1: 0.1855
Epoch 004 | LR: 0.001000 | Train Loss: 1.8768 | Train Acc: 0.2447 | Train F1: 0.2107 | Val Loss: 1.9125 | Val Acc: 0.2370 | Val F1: 0.1855
Epoch 005 | LR: 0.001000 | Train Loss: 1.8753 | Train Acc: 0.2417 | Train F1: 0.2128 | Val Loss: 1.9034 | Val Acc: 0.2345 | Val F1: 0.1907
Epoch 006 | LR: 0.001000